# RCEA Worked Example: Privacy/Recruitment

This notebook walks through the full RCEA passport generation pipeline for the
privacy/recruitment finding, demonstrating role-conditioned views for DPO, CISO,
AI Lead, and Executive.


In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from rcea.models import EvidenceFinding, ContextPack, RoleProfile, RulePack
from rcea.passport import generate_passport
from rcea.rendering import render_passport_markdown

BASE = Path('../examples/privacy_recruitment')
finding = EvidenceFinding.model_validate_json((BASE / 'finding.json').read_text())
context  = ContextPack.model_validate_json((BASE / 'context_pack.json').read_text())
roles_data = json.loads((BASE / 'role_profiles.json').read_text())
roles = {r['role_id']: RoleProfile.model_validate(r) for r in roles_data}
rule_pack = RulePack.model_validate_json((BASE / 'rule_pack.json').read_text())

print(f'Finding:  {finding.metric} = {finding.value}  (threshold: {finding.threshold})')
print(f'Severity: {finding.severity.value.upper()}')
print(f'Method:   {finding.method}')
print(f'Roles:    {list(roles.keys())}')

In [ ]:
# Generate passports for all roles
passports = {role_id: generate_passport(finding, role, context, rule_pack)
             for role_id, role in roles.items()}

# Display RCEA score table
score_keys = ['material_relevance', 'epistemic_warrant', 'normative_alignment',
              'interpretive_fit', 'decision_actionability', 'limitation_propagation',
              'audit_traceability', 'overall']

header = f"{'Role':<14}" + "".join(f"{k[:6]:>9}" for k in score_keys)
print(header)
print('-' * len(header))
for role_id, passport in passports.items():
    scores = passport.rcea_scores
    row = f"{role_id:<14}" + "".join(
        f"{scores.get(k, float('nan')):>9.4f}" for k in score_keys
    )
    print(row)

In [ ]:
# Show the DPO passport as Markdown
from IPython.display import Markdown
Markdown(render_passport_markdown(passports['dpo']))

In [ ]:
# Compare visible fields across roles
print('Visible fields per role:')
for role_id, passport in passports.items():
    fields = sorted(passport.visible_fields.keys())
    print(f'  {role_id:<14}: {fields}')

In [ ]:
# Show suppression logs
print('Suppression logs per role:')
for role_id, passport in passports.items():
    print(f'\n  {role_id}:')
    for entry in passport.suppression_log:
        print(f'    [{entry.field}] {entry.rationale}')